# Лекция 15. Чистка транзакционного датасета

In [1]:
import numpy as np
import pandas as pd

## Загрузка

In [2]:
df = pd.read_csv('l13_transactions_raw.csv')
print("Размер:", df.shape)
df.head()

Размер: (50120, 11)


,Transaction ID,Card ID,DateTime,Merchant,Category,Region,Amount (UZS),Status,Is Online,MCC,Cashback (UZS)
0,1000000,UZC-00372,2026-05-25T14:27:02,Apteka,Аптека,Наманган,131900,success,N,5912,NaN
1,1000001,UZC-00367,2026-05-25T12:03:40,Yandex Taxi,Транспорт,Ташкент,85500,success,N,4121,998.0
2,1000002,UZC-02063,2026-05-25T03:07:55,Texnomart,Электроника,Фергана,370700,success,Y,5732,7084.0
3,1000003,UZC-01621,2026-05-25T17:03:45,Evos,Кафе,Бухара,41700,success,N,5814,0.0
4,1000004,UZC-00473,2026-05-25T09:07:55,Apteka,Аптека,Нукус,154 700,success,Y,5912,NaN


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50120 entries, 0 to 50119
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Transaction ID  50120 non-null  int64  
 1   Card ID         50120 non-null  str    
 2    DateTime       50120 non-null  str    
 3   Merchant        50120 non-null  str    
 4   Category        50120 non-null  str    
 5   Region          50020 non-null  str    
 6   Amount (UZS)    50120 non-null  str    
 7   Status          50120 non-null  str    
 8   Is Online       50120 non-null  str    
 9   MCC             50120 non-null  int64  
 10  Cashback (UZS)  32575 non-null  float64
dtypes: float64(1), int64(2), str(8)
memory usage: 4.2 MB


In [4]:
df.isnull().sum()

Transaction ID        0
Card ID               0
 DateTime             0
Merchant              0
Category              0
Region              100
Amount (UZS)          0
Status                0
Is Online             0
MCC                   0
Cashback (UZS)    17545
dtype: int64

## Проблемы в данных

| # | Столбец | Проблема |
|---|---|---|
| 1 | Все | Пробелы и `(UZS)` в именах столбцов |
| 2 | `Amount (UZS)` | Текст: `'154 700'`, `'181,000'`, `'н/д'` |
| 3 | Вся таблица | 120 дубликатов |
| 4 | `DateTime` | Дата как текст |
| 5 | `Is Online` | `'Y'`/`'N'` вместо bool |
| 6 | `Region` | 32 варианта написания |
| 7 | `Region` | 100 NaN |
| 8 | `Cashback (UZS)` | NaN != 0 (разный смысл) |
| 9 | `Amount (UZS)` | 15 выбросов > 3 млн |
| 10 | `MCC` | Технический столбец, не нужен |

## Шаг 1. Имена столбцов

`strip`, `lower`, убрать `(uzs)`, пробелы → `_`

## Шаг 2. Дубликаты

## Шаг 3. Сумма в число

Форматы: `'154 700'`, `'181,000'`, `'н/д'`. Убрать разделители, потом `to_numeric(errors="coerce")`.

## Шаг 4. Дата

## Шаг 5. Is Online → bool

## Шаг 6. Регион: написание

## Шаг 7. Регион: NaN

## Шаг 8. Cashback

NaN = данные не пришли. 0 = нет программы лояльности. `fillna(0)` смешивает два разных случая.

## Шаг 9. Выбросы (amount > 3 млн)

## Шаг 10. Убрать MCC

## Финальная проверка и сохранение

## Итог

| | Сырой файл | Чистый файл |
|---|---|---|
| Строк | 50 120 | ~49 840 |
| Столбцов | 11 | 9 |
| NaN | есть | 0 |
| Дубликаты | 120 | 0 |
| Варианты региона | 32 | 9 |

## Regex: краткий разбор

Шаг 3 использовал `str.replace(r"[ ,]", "", regex=True)`.
Regex (регулярные выражения): язык паттернов для поиска и замены текста.

| Паттерн | Что значит | Пример |
|---|---|---|
| `[ ,]` | символ-класс: пробел ИЛИ запятая | `r"[ ,]"` |
| `\d` | любая цифра 0-9 | `r"\d+"` |
| `\D` | всё что не цифра | `r"\D"` |
| `\s` | пробел, таб, перенос строки | `r"\s+"` |
| `[^\d]` | исключение: всё кроме цифр | `r"[^\d]"` |
| `+` | один или более | `r"\d+"` |
| `*` | ноль или более | `r"\s*"` |

Строка должна быть `r"..."` (raw string), иначе Python обработает `\d` как спецсимвол.

In [ ]:
s = pd.Series(["154 700", "181,000", "45 000 so'm", "н/д"])

# убрать пробел и запятую
print(s.str.replace(r"[ ,]", "", regex=True).tolist())

# убрать всё кроме цифр (более агрессивно)
print(s.str.replace(r"[^\d]", "", regex=True).tolist())

# результат: первый паттерн оставляет "н/д", второй его тоже зачищает
# to_numeric(errors="coerce") превратит оба нечитаемых варианта в NaN